# 16.3 Rules for columnwise formulations

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The algebraic description of an AMPL constraint can be written in any of the following
ways:

```ampl
arith-expr <= arith-expr
arith-expr = arith-expr
arith-expr >= arith-expr
const-expr <= arith-expr <= const-expr
const-expr >= arith-expr >= const-expr
```

Each const-expr must be an arithmetic expression not containing variables, while an
arith-expr may be any valid arithmetic expression — though it must be linear in the variables
([Section 8.2](../08/8_2_linear_expressions.ipynb)) if the result is to be a linear program. To permit a columnwise formulation,
one of the arith-exprs may be given as:

```ampl
to_come
to_come + arith-expr
arith-expr + to_come
```

Most often a "template" constraint of this kind consists, as in our examples, of
`to_come`, a relational operator and a const-expr; the constraint's linear terms are all provided
in subsequent `var` declarations, and `to_come` shows where they should go. If the
template constraint does contain variables, they must be from previous `var` declarations,
and the model becomes a sort of hybrid between row-wise and columnwise forms.

The expression for an objective function may also incorporate `to_come` in one of the
ways shown above. If the objective is a sum of linear terms specified entirely by subsequent
`var` declarations, as in our examples, the expression for the objective is just
`to_come` and may be omitted.

In a `var` declaration, constraint coefficients may be specified by one or more phrases
consisting of the keyword `coeff`, an optional indexing expression, a constraint name,
and an arith-expr. If an indexing expression is present, a coefficient is generated for each
member of the indexing set; otherwise, one coefficient is generated. The indexing
expression may also take the special form {if logical-expr} as seen in [Section 8.4](../08/8_4_variables.ipynb) or 15.3,
in which case a coefficient is generated only if the logical-expr evaluates to true.
Our simple examples have required just one `coeff` phrase in each `var` declaration, but

```ampl
set CITIES;
set LINKS within (CITIES cross CITIES);
set PRODS;
param supply {CITIES,PRODS} >= 0;  # amounts available at cities
param demand {CITIES,PRODS} >= 0;  # amounts required at cities
       check {p in PRODS}:
              sum {i in CITIES} supply[i,p] = sum {j in CITIES} demand[j,p];
param cost {LINKS,PRODS} >= 0;     # shipment costs/1000 packages
param capacity {LINKS,PRODS} >= 0; # max packages shipped
param cap_joint {LINKS} >= 0;      # max total packages shipped/link
minimize Total_Cost;
node Balance {k in CITIES, p in PRODS}:
net_in = demand[k,p] - supply[k,p];
subject to Multi {(i,j) in LINKS}:
to_come <= cap_joint[i,j];
arc Ship {(i,j) in LINKS, p in PRODS} >= 0, <= capacity[i,j,p],
from Balance[i,p], to Balance[j,p],
coeff Multi[i,j] 1.0,
obj Total_Cost cost[i,j,p];
```

**[Figure 16-6](./16_3_rules_for_columnwise_formulations.ipynb#fig-16-6):** Columnwise formulation of [Figure 15-13](#missing) <!--- xTODO missing reference ---> (netmcol.mod).

in general a separate `coeff` phrase is needed for each different indexed collection of
constraints in which a variable appears.

Objective function coefficients may be specified in the same way, except that the keyword
obj is used instead of `coeff`.

The obj phrase in a `var` declaration is the same as the obj phrase used in `arc` declarations
for networks ([Section 15.4](../15/15_4_rules_for_node_and_arc_declarations.ipynb)). The constraint coefficients for the network variables
defined by an `arc` declaration are normally given in from and to phrases, but
`coeff` phrases may be present as well; they can be useful if you want to give a columnwise
description of "side" constraints that apply in addition to the balance-of-flow constraints.
As an example, [Figure 16-6](./16_3_rules_for_columnwise_formulations.ipynb#fig-16-6) shows how a `coeff` phrase can be used to rewrite
the multicommodity flow model of [Figure 15-13](#missing) <!--- xTODO missing reference ---> in an entirely columnwise manner.

**Bibliography**

Gerald Kahan, "Walking Through a Columnar Approach to Linear Programming of a Business."
Interfaces 12, 3 (1982) pp. 32-39. A brief for the columnwise approach to linear programming,
with a small example.